In [1]:
import pandas as pd
for f in ["movies_metadata.csv","credits.csv","keywords.csv","ratings.csv"]:
        df = pd.read_csv(f, nrows=3, low_memory=False)
        print(f, "->", list(df.columns))
        print(df.head(2))
        print("---")

movies_metadata.csv -> ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']
   adult                              belongs_to_collection    budget  \
0  False  {'id': 10194, 'name': 'Toy Story Collection', ...  30000000   
1  False                                                NaN  65000000   

                                              genres  \
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   

                               homepage    id    imdb_id original_language  \
0  http://toystory.disney.com/toy-story   862  tt0114709                en   
1                                   NaN  8844  tt0113497                en   

  original_tit

In [3]:
import ast
m = pd.read_csv("movies_metadata.csv", low_memory=False)[["id","title","genres","runtime","release_date"]]
m = m[m["id"].apply(lambda x: str(x).isdigit())]
m["id"] = m["id"].astype(int)
def genre_names(s):
        try: genre_list = ast.literal_eval(s)
        except Exception: genre_list = []
        names = [g["name"] for g in genre_list]
        return ", ".join(names) if names else "Unknown"
    

In [7]:
c = pd.read_csv("credits.csv", nrows=5)
def get_director(s):
        try: crew_list = ast.literal_eval(s)
        except Exception: crew_list = []
        directors = [p.get("name","Unknown") for p in crew_list if p.get("job") == "Director"]
        return directors[0] if directors else "Unknown"
c["director"] = c["crew"].apply(get_director)
print(c[["id","director"]])

      id         director
0    862    John Lasseter
1   8844     Joe Johnston
2  15602    Howard Deutch
3  31357  Forest Whitaker
4  11862    Charles Shyer


In [8]:
r = pd.read_csv("ratings.csv", usecols=["movieId","rating"], nrows=200000)
avg = r.groupby("movieId")["rating"].mean().round(1).reset_index()
avg = avg.rename(columns={"movieId":"id","rating":"avg_rating"})
print(avg.head())
merged = m.merge(c[["id","director"]], on="id", how="left").merge(avg, on="id", how="left")
print(merged[["id","title","genre","runtime","director","avg_rating"]].head(10))

   id  avg_rating
0   1         3.9
1   2         3.2
2   3         3.1
3   4         2.7
4   5         3.0


KeyError: "['genre'] not in index"

In [9]:
m["genre"] = m["genres"].apply(genre_names)
merged = m.merge(c[["id","director"]], on="id", how="left").merge(avg, on="id", how="left")
print(merged[["id","title","genre","runtime","director","avg_rating"]].head(10))

      id                        title                             genre  \
0    862                    Toy Story         Animation, Comedy, Family   
1   8844                      Jumanji        Adventure, Fantasy, Family   
2  15602             Grumpier Old Men                   Romance, Comedy   
3  31357            Waiting to Exhale            Comedy, Drama, Romance   
4  11862  Father of the Bride Part II                            Comedy   
5    949                         Heat    Action, Crime, Drama, Thriller   
6  11860                      Sabrina                   Comedy, Romance   
7  45325                 Tom and Huck  Action, Adventure, Drama, Family   
8   9091                 Sudden Death       Action, Adventure, Thriller   
9    710                    GoldenEye       Adventure, Action, Thriller   

   runtime         director  avg_rating  
0     81.0    John Lasseter         3.0  
1    104.0     Joe Johnston         NaN  
2    101.0    Howard Deutch         NaN  
3    1